# Big Notebook Part 9: Real Neurons from the Allen Cell Types Database
This time the data is **real**: electrophysiological recordings of **2,333 single neurons** from mouse and human cortex, published by the Allen Institute for Brain Science (Allen Cell Types Database, downloaded via the Allen Brain Map API).

Every row is one neuron. Among other things you'll find:
- the **species** (mouse or human) and the **dendrite type** (`spiny` ≈ mostly excitatory cells, `aspiny` ≈ mostly inhibitory interneurons)
- the cortical **layer** and **brain area**
- electrophysiological features: **resting potential** (`vrest`), **input resistance** (`ri`), **membrane time constant** (`tau`), **average firing rate**, and the **upstroke/downstroke ratio** of the action potential (a measure of spike width)
- for human cells: **age** and **disease** of the donor (the tissue comes from epilepsy or tumour surgery)

Real data is messy: long column names, missing values, numbers stored as text. That's part of the exercise.

10 tasks, about 60–75 minutes. Focus: `groupby` (Part 9), plus your core patterns.
- 🔍 = Spot the bug
- 🧠 = DataFrame, Series or a single value?
- Remember the **print rule** and the **assignment rule**.

---
## Setup
Make sure `allen_cells.csv` is in your `progress` folder.

In [2]:
import pandas as pd
from pandas.core.interchange import column
# Path to the data file – adjust to where you saved allen_cells.csv
PFAD = "/Users/sven/Desktop/Udemy/progress/allen_cells.csv"

cells = pd.read_csv(PFAD)
cells.head()

,line_name,specimen__id,specimen__name,specimen__hemisphere,structure__id,structure__name,structure__acronym,structure_parent__id,structure_parent__acronym,structure__layer,...,ephys_thumb_path,ephys_inst_thresh_thumb_path,donor__age,donor__sex,donor__disease_state,donor__race,donor__years_of_seizure_history,donor__species,donor__id,donor__name
0,NaN,525011903,H16.03.003.01.14.02,right,12113,"""frontal lobe""",FroL,12113,FroL,3,...,/api/v2/well_known_file_download/529903140,/api/v2/well_known_file_download/529903142,25 yrs,Male,epilepsy,White or Caucasian,9.0,Homo Sapiens,524848408,H16.03.003
1,Oxtr-T2A-Cre,565871768,Oxtr-2A-Cre;Ai14-293837.03.01.02,left,778,"""Primary visual area, layer 5""",VISp5,385,VISp,5,...,/api/v2/well_known_file_download/566738214,/api/v2/well_known_file_download/566738219,NaN,NaN,NaN,NaN,NaN,Mus musculus,563340157,Oxtr-T2A-Cre;Ai14-293837
2,Pvalb-IRES-Cre,469801138,Pvalb-IRES-Cre;Ai14-170927.05.02.01,right,721,"""Primary visual area, layer 4""",VISp4,385,VISp,4,...,/api/v2/well_known_file_download/487618917,/api/v2/well_known_file_download/491380580,NaN,NaN,NaN,NaN,NaN,Mus musculus,319194354,Pvalb-IRES-Cre;Ai14(IVSCC)-170927
3,NaN,528642047,H16.06.009.01.02.06.05,left,12141,"""middle temporal gyrus""",MTG,12141,MTG,5,...,/api/v2/well_known_file_download/660970973,/api/v2/well_known_file_download/660970975,48 yrs,Female,epilepsy,unknown,33.0,Homo Sapiens,528574320,H16.06.009
4,Slc32a1-T2A-FlpO|Vipr2-IRES2-Cre,605889373,Vipr2-IRES2-Cre;Slc32a1-T2A-FlpO;Ai65-337419.0...,right,821,"""Primary visual area, layer 2/3""",VISp2/3,385,VISp,2/3,...,/api/v2/well_known_file_download/657612184,/api/v2/well_known_file_download/657612186,NaN,NaN,NaN,NaN,NaN,Mus musculus,605622695,Vipr2-IRES2-Cre;Slc32a1-T2A-FlpO;Ai65-337419


---
## Tasks

**1 ⭐ First look:** How many rows and columns does `cells` have? And how many neurons come from each species (`donor__species`)?

In [ ]:
# cells.info() # 53 columns and 2333 rows
cells['donor__species'].value_counts()

**2 ⭐ Keep only what you need:** 54 columns are too many. Create a new DataFrame `zellen` that contains **only** the columns in the dictionary below, **renamed** to the short names.

```python
namen = {
    "donor__species": "species",
    "tag__dendrite_type": "dendrite_type",
    "structure__layer": "layer",
    "structure_parent__acronym": "area",
    "ef__vrest": "vrest_mV",
    "ef__ri": "ri_MOhm",
    "ef__tau": "tau_ms",
    "ef__avg_firing_rate": "firing_rate_Hz",
    "ef__upstroke_downstroke_ratio_long_square": "updown_ratio",
    "donor__age": "age",
    "donor__disease_state": "disease",
}
```

*Tip:* `list(namen)` gives you the list of the old column names.

In [ ]:
namen = {
    "donor__species": "species",
    "tag__dendrite_type": "dendrite_type",
    "structure__layer": "layer",
    "structure_parent__acronym": "area",
    "ef__vrest": "vrest_mV",
    "ef__ri": "ri_MOhm",
    "ef__tau": "tau_ms",
    "ef__avg_firing_rate": "firing_rate_Hz",
    "ef__upstroke_downstroke_ratio_long_square": "updown_ratio",
    "donor__age": "age",
    "donor__disease_state": "disease",
}


zellen = (cells[list(namen)].rename(columns=namen))
zellen

**3 ⭐ Readable species names:** Replace `"Mus musculus"` with `"mouse"` and `"Homo Sapiens"` with `"human"` in the column `species`. Check with `value_counts()`.

In [ ]:
zellen['species'] = zellen['species'].replace({'Mus musculus' : 'mouse', 'Homo Sapiens' : 'human'})
zellen['species'].value_counts()

**4 ⭐⭐ Human vs. mouse:** Compute the **mean** `vrest_mV`, `ri_MOhm` and `tau_ms` per `species`, rounded to 1 decimal. Which species has the longer membrane time constant?

In [ ]:
zellen.groupby('species').agg({'vrest_mV':'mean', 'ri_MOhm':'mean', 'tau_ms':'mean'})

**5 ⭐⭐ Who fires fastest?** Compute the mean `firing_rate_Hz` for every combination of `species` and `dendrite_type`. Turn `dendrite_type` into columns with `unstack` and round to 1 decimal.

In [ ]:
zellen.groupby(['species', 'dendrite_type'])['firing_rate_Hz'].mean().round(1).unstack('dendrite_type')


**6 🔍 `size` or `count`?** Someone wants to know for how many neurons per species a **firing rate** is available, and writes:

```python
zellen.groupby("species").size()
```

Why is this the wrong answer? Fix it. How many neurons have **no** firing rate?

In [ ]:
zellen.groupby('species').size() - zellen.groupby('species')['firing_rate_Hz'].count()

**7 ⭐⭐ Several statistics:** Per `dendrite_type`, show in one table: mean and maximum of `firing_rate_Hz` and the **median** of `ri_MOhm`. Round to 1 decimal.

In [ ]:
zellen.groupby('dendrite_type').agg({'firing_rate_Hz':['mean', 'max'], 'ri_MOhm' : 'median'}).round(1)

**8 ⭐⭐⭐ Function + `apply`: spike width.** Fast-spiking interneurons have **narrow** action potentials. A common rule of thumb: an upstroke/downstroke ratio **below 2** means a narrow spike.

Write a function `spike_typ(zeile)` that returns `"narrow"` if `updown_ratio` is below 2, otherwise `"broad"`. Store it as column `spike`. Then count the spike types **per `dendrite_type`**. 🧠 Inside the function, what is `zeile["updown_ratio"]`?

In [ ]:
def spike_typ(zeile):
    if zeile['updown_ratio'] < 2:
        return 'narrow'
    else:
        return 'broad'

zellen['spike'] = zellen.apply(spike_typ, axis = 'columns')

zellen.groupby('dendrite_type')['spike'].value_counts()

# zeile["updown_ratio"] is a series


**9 ⭐⭐⭐ Human donors – age as text:** (a) Create a DataFrame `mensch` with **only the human cells**. (b) The column `age` contains text like `"25 yrs"`. Create a numeric column `age_years` with `split`, `.str[0]` and `astype(int)`. (c) Write a function that returns `"under 40"` or `"40 and older"`, store it as `age_group`, and compare mean `vrest_mV` and `ri_MOhm` between the age groups.

🧠 Why would `astype(int)` fail if you did (b) on `zellen` instead of `mensch`?

In [ ]:
mensch = zellen[zellen['species'] == 'human']
mensch['age_years'] = mensch['age'].str.split(' ').str[0].astype(int)
mensch['age_years']

def age(zeile):
    if zeile['age_years'] < 40:
        return 'under 40'
    else:
        return '40 and older'

mensch['age_group'] = mensch.apply(age, axis = 'columns')
mensch.groupby('age_group')[['vrest_mV', 'ri_MOhm']].mean().round(1)



**10 ⭐⭐ Loop over the groups:** For the human cells, loop over `mensch.groupby("disease")` and print for each disease: the name, the number of neurons and the mean `firing_rate_Hz` (rounded to 1 decimal).

In [ ]:
for name, gruppe in mensch.groupby('disease'):
    print(name, len(gruppe), gruppe['firing_rate_Hz'].mean()).round(1)



# for sector_name, sector_df in sectors:
#     print(sector_name, len(sector_df))

---
✅ **Done?** Restart Kernel and Run All, then let me know! Next step: a short README and your first GitHub upload. 🚀